In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
import torch

print(torch.version.cuda)

12.8


In [3]:
!pip install -q \
transformers \
datasets \
accelerate \
peft \
trl \
bitsandbytes \
sentencepiece

In [ ]:

import transformers
import peft
import datasets
import trl

print("Everything installed!")

In [ ]:
import os

os.makedirs("dataset", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes sentencepiece

In [ ]:
import torch

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

from datasets import load_dataset

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-2B"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageTextToText.from_pretrained(MODEL_NAME)

In [ ]:
from transformers import AutoModelForImageTextToText

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
model.eval()

In [ ]:
!ls

In [ ]:
!find dataset -maxdepth 2

In [ ]:
!ls dataset/images | head

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image = Image.open("dataset/images/0.jpg")

plt.imshow(image)
plt.axis("off")

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse("dataset/annotations.xml")
root = tree.getroot()

print(root.tag)

In [ ]:
for child in root[:2]:
    print(child.tag, child.attrib)

In [ ]:
!find dataset -maxdepth 2

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse("dataset/annotations.xml")
root = tree.getroot()

print(ET.tostring(root[0], encoding="unicode"))

In [ ]:
import pandas as pd

df = pd.read_csv("dataset/receipts.csv")

df.head()

In [ ]:
print(df.columns)

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse("dataset/annotations.xml")
root = tree.getroot()

print(root.tag)

for child in root:
    print(child.tag)

In [ ]:
print(ET.tostring(root, encoding="unicode")[:2000])

In [ ]:
image = root.findall("image")[0]

print(ET.tostring(image, encoding="unicode"))

In [ ]:
image = root.findall("image")[0]

# print(ET.tostring(image, encoding="unicode"))

# Final

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image
import pandas as pd

In [ ]:
tree = ET.parse("dataset/annotations.xml")
root = tree.getroot()

images = root.findall("image")

print(f"Found {len(images)} annotated receipts.")

In [ ]:
def image_to_markdown(image_element):
    shop = ""
    total = ""
    date = ""
    items = []

    for box in image_element.findall("box"):
        label = box.attrib["label"]
        text = box.find("attribute").text.strip()

        if label == "shop":
            shop = text

        elif label == "item":
            items.append(text)

        elif label == "total":
            total = text

        elif label == "date_time":
            date = text

    markdown = "# Receipt\n\n"

    if shop:
        markdown += f"## Shop\n{shop}\n\n"

    if items:
        markdown += "## Items\n"
        for item in items:
            markdown += f"- {item}\n"
        markdown += "\n"

    if total:
        markdown += f"## Total\n{total}\n\n"

    if date:
        markdown += f"## Date\n{date}\n"

    return markdown

In [ ]:
training_examples = []

for image in images:

    image_path = Path("dataset") / image.attrib["name"]

    markdown = image_to_markdown(image)

    training_examples.append(
        {
            "image_path": str(image_path),
            "markdown": markdown
        }
    )

print("Number of samples:", len(training_examples))

In [ ]:
print(training_examples[0]["image_path"])

print()

print(training_examples[0]["markdown"])

In [ ]:
import matplotlib.pyplot as plt

img = Image.open(training_examples[0]["image_path"])

plt.imshow(img)
plt.axis("off")

print(training_examples[0]["markdown"])

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_list(training_examples)

hf_dataset

In [ ]:
from PIL import Image

def preprocess(example):
    image = Image.open(example["image_path"]).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": "Extract all text from this receipt and format it as Markdown.",
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt",
    )

    labels = processor.tokenizer(
        example["markdown"],
        return_tensors="pt",
    )["input_ids"][0]

    result = {
        k: v.squeeze(0).tolist()
        for k, v in inputs.items()
    }
    result["labels"] = labels.tolist()

    return result

In [ ]:
processed_dataset = hf_dataset.map(preprocess)

In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
!pip install -U torchao

In [ ]:
model.print_trainable_parameters()

In [ ]:
train_test = hf_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = train_test["train"]
val_dataset = train_test["test"]

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_receipt_lora",

    num_train_epochs=10,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=2e-4,

    logging_steps=1,

    eval_strategy="epoch",
    save_strategy="epoch",

    fp16=True,

    remove_unused_columns=False,

    report_to="none",
)

In [ ]:
train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)

In [ ]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "mm_token_type_ids",
    "pixel_values",
    "image_grid_thw",
    "labels",
]

columns_to_remove = [
    col for col in train_dataset.column_names
    if col not in columns_to_keep
]

train_dataset = train_dataset.remove_columns(columns_to_remove)
val_dataset = val_dataset.remove_columns(columns_to_remove)

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

def data_collator(features):
    input_ids = [
        torch.tensor(f["input_ids"], dtype=torch.long)
        for f in features
    ]

    attention_mask = [
        torch.tensor(f["attention_mask"], dtype=torch.long)
        for f in features
    ]

    mm_token_type_ids = [
        torch.tensor(f["mm_token_type_ids"], dtype=torch.long)
        for f in features
    ]

    labels = [
        torch.tensor(f["labels"], dtype=torch.long)
        for f in features
    ]

    pixel_values = [
        torch.tensor(f["pixel_values"], dtype=torch.float)
        for f in features
    ]

    image_grid_thw = [
        torch.tensor(f["image_grid_thw"], dtype=torch.long)
        for f in features
    ]

    batch = {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=processor.tokenizer.pad_token_id,
        ),

        "attention_mask": pad_sequence(
            attention_mask,
            batch_first=True,
            padding_value=0,
        ),

        "mm_token_type_ids": pad_sequence(
            mm_token_type_ids,
            batch_first=True,
            padding_value=0,
        ),

        "labels": pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        ),

        "pixel_values": torch.cat(pixel_values, dim=0),

        "image_grid_thw": torch.stack(image_grid_thw),
    }

    return batch

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
)

In [ ]:
trainer.train()